# 11. Графики события и встроенный LightHit viewer

`viewer.json` — переносимые данные: геометрия, три компоненты, интегральные
заряды и временные бины. `viewer.html` — автономное представление с Plotly;
оно не пересчитывает транспорт и не меняет знаки.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import IFrame,display
from lighthit.viewer import load_event_result
path=Path(".build/course/shower-viewer.json")
if not path.exists():raise RuntimeError("Сначала выполните 10_axial_event.ipynb")
data=load_event_result(path);event=data["events"][0]
print(data["schema"],event["label"],len(data["detector"]["positions_m"]),"ОМ")


## 11.1. Контроль до визуализации

Интерфейс не является проверкой физики. До его открытия сверяем формы,
заряд ω=0, сумму временного окна и отрицательную массу. Расхождение полного
заряда и окна нельзя исправлять перенормировкой.


In [ ]:
c=np.asarray(event["components"]);q=np.asarray(event["charge_components"])
edges=np.asarray(data["readout"]["relative_time_edges_ns"]);window=c.sum(axis=1)
print("shape",c.shape,"Q total",q.sum(),"window",window.sum(),
      "negative |mass| fraction",np.abs(c[c<0]).sum()/np.abs(c).sum())
assert c.shape==(len(q),len(edges)-1,3) and np.isfinite(c).all()


## 11.2. Две статические проекции тех же данных


In [ ]:
total=q.sum(axis=1);bright=np.argmax(np.abs(total));centres=(edges[:-1]+edges[1:])/2
positions=np.asarray(data["detector"]["positions_m"])
fig,ax=plt.subplots(1,2,figsize=(12,4))
scale=max(np.max(np.abs(total))*1e-6,1e-300)
sc=ax[0].scatter(positions[:,0],positions[:,2],c=np.sign(total)*np.log10(1+np.abs(total)/scale),cmap="coolwarm")
ax[0].set(xlabel="x, м",ylabel="z, м",title="sign·log интегрального заряда");fig.colorbar(sc,ax=ax[0])
ax[1].step(centres,c[bright].sum(axis=1),where="mid")
ax[1].set(xlabel="время от фронта, нс",ylabel=data["units"]["signal"]+" / бин",title=f"ОМ {bright}")
fig.tight_layout()


## 11.3. Тот же результат в интерактивном viewer

Доступны выбор события, кластера и порядка, интегральный заряд, временная
гистограмма, heatmap всех ОМ, текущий или накопленный кадр и анимация.
Соседние бины можно объединять ×2/×4: это подавляет видимый ringing, но
не меняет сохранённые данные и не восстанавливает частоты выше cutoff.


In [ ]:
viewer=Path(".build/course/shower-viewer.html").resolve()
display(IFrame(src="shower-viewer.html",width="100%",height=900))
print("Открыть отдельно:",viewer)


## 11.4. Производственный запуск

Для четырёх локальных G4-файлов и лазера:

```bash
python scripts/run_event_viewer.py --output .build/event-viewer --threads 4
```

Команда использует общий Numba-кэш, порог 0.01 фэ с точным ω=0 guard,
обрабатывает события последовательно, сохраняет NPZ каждого события и
создаёт общий `viewer.html`. Реальные G4-файлы
и результаты под `.build/` не входят в исходники.

### Задания

1. Выбрать порядок ≥2 и найти ОМ с максимальным поздним хвостом.
2. Сравнить интегральный заряд и заряд окна для мюона.
3. Объяснить, почему размер маркера использует модуль, а цвет сохраняет знак.
